
Dataset Link: https://data.mendeley.com/datasets/th7fztbrv9/11

In [33]:
import pandas as pd
from sklearn import preprocessing
from sklearn.impute import KNNImputer
#import re

In [ ]:
df = pd.read_excel('Data/SD1.xlsx')
df.head()

In [45]:
x = df.drop(['TYPE', 'SUBJECT_ID'], axis=1) #Input_Data
y = df['TYPE'] # Target

Okay, so next step was going to be normalizing these features, as not to skew the results, but we have encountered a problem. Our raw training data isn't integer, but string in some places, so we have to fix that

In [ ]:
is_str = x['AFP'].apply(lambda cell: isinstance(cell, str))
combined_patterns = r'\t|>|<|>.\t'
x.loc[is_str, 'AFP'] = x.loc[is_str, 'AFP'].str.replace(combined_patterns, '', regex=True)
x['AFP'].astype(float)

Above is how we would do it for individual column, now we extrapolate it to entire dataset. One thing to keep in mind is that since we are removing greater than and lesser than symbol, technically we are making the data less accurate, because it is possible that the values are way higher but the sampler taker had no way of knowing the exact value

In [46]:
combined_patterns = r'\t|>|<'
x = x.replace(combined_patterns, '', regex=True)
x = x.astype(float)

We have to split our dataset into train and test in the beginning itself, because any normalization will leak out the underlying distribution of the dataset

In [47]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

Now we finally normalize our data. Here we are going to use Min-Max scaler. Formula is (x - x_min)/(x_max - x_min)

Note: We fit_transform() our train data, but only transform() our test data. It means that we scale the test data according to the training data

In [ ]:
min_max_scaler = preprocessing.MinMaxScaler()
x_train_scale = min_max_scaler.fit_transform(x_train)
x_test_scale = min_max_scaler.transform(x_test)

Since we are using KNN Imputer (which is distance based), we need to normalize our inputs before using it
Given below is a single column implementation just for demonstration

In [ ]:
print(x['AFP'].isnull().value_counts())
imputer = KNNImputer(n_neighbors=5)
single_col = x[['AFP']]
x['AFP'] = imputer.fit_transform(single_col)

x['AFP'].isnull().value_counts()


Full Column imputation

In [ ]:
imputer = KNNImputer()
x_train_impute = imputer.fit_transform(x_train_scale)
x_test_impute = imputer.transform(x_test_scale)